# TEST FOR ANYLOGISTIX API
THe API info is here: https://anylogistix.help/api/python/readme.html

## PHASE 1  Connect API, get scenarios, select 1, run it and collect the simulated data

## Install the dependencies
It assumes that the openapi_client is in the local directory ./openapi-python-client-3.3.1

In [3]:
import sys
import os
# This is my personal directory
os.chdir(r"C:\Users\Luis\Downloads\TFG\API")
#!{sys.executable} -m pip install "pydantic>=2.10,<3"
!{sys.executable} -m pip install "pydantic<2,>=1.10.5"
!{sys.executable} -m pip install -e ./openapi-python-client-3.3.1


Obtaining file:///C:/Users/Luis/Downloads/TFG/API/openapi-python-client-3.3.1
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for openapi-client (pyproject.toml): started
  Building editable for openapi-client (pyproject.toml): finished with status 'done'
  Created wheel for openapi-client: filename=openapi_client-1.0.0-0.editable-py3-none-any.whl size=3042 sha256=b3059e5926115896a4ed12a11556076e447d928d34fe080e3b7515423230f201
  Stored in directory: C:\Users\Luis\AppData\Local\Temp\pip-ephem-wheel-cache-6vn1tqk8\wheels\96\67\

In [4]:
import openapi_client
from openapi_client.rest import ApiException
from pprint import pprint
import urllib3

import shutil #For manipulating excel files
import re # For cleaning strings

In [5]:
# SERVER_IP="192.168.64.159"
# SERVER_IP="192.168.67.110"
SERVER_IP="alxserver.aut.uah.es"
SERVER_URL=f"https://{SERVER_IP}:443/api/v1"

# This is my personal APY KEY
API_KEY="c184f1ab-9f13-484c-a1c1-3d543502da6e"

SERVER_URL

'https://alxserver.aut.uah.es:443/api/v1'

## PART I: CHECK CONNECTIVIY AND OPERATIONAL STATUS

In [6]:
# Defining the host is optional and defaults to https://server_address:port/api/v1
# See configuration.py for a list of all supported configuration parameters.
configuration = openapi_client.Configuration(
    host = SERVER_URL
)
# The client must configure the authentication and authorization parameters
# in accordance with the API server security policy.
# Examples for each auth method are provided below, use the example that
# satisfies your auth use case.
# Configure API key authorization: ApiKey
configuration.api_key['ApiKey'] = API_KEY

# ALVARO: SUPER IMPORTANTE PARA PODER TRABAJAR CON CERTIFICADOS AUTO-FIRMADOS
configuration.verify_ssl = False
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
with openapi_client.ApiClient(configuration) as api_client:
    # Create an instance of the API class
    api_instance = openapi_client.OpenApiApi(api_client)

### Obtain current user data

In [7]:
try:
    # Gets current user data.
    api_response = api_instance.get_current_user()
    print("The response of OpenApiApi->get_current_user:\n")
    pprint(api_response)
except Exception as e:
    print("Exception when calling OpenApiApi->get_current_user: %s\n" % e)

The response of OpenApiApi->get_current_user:

ApiUserData(email='adrian.encabo@edu.uah.es', first_name='adrian.encabo@edu.uah.es', id=403, last_name='', username='adrian.encabo@edu.uah.es')


## PART II: OBTAIN PROJECTS, SCENARIOS, EXECUTIONS

### Obtain the list of currently available projects

In [8]:
try:
    # Returns a list of projects that the user has access to.
    api_response = api_instance.get_projects()
    print("The response of OpenApiApi->get_projects:\n")
    pprint(api_response)
except Exception as e:
    print("Exception when calling OpenApiApi->get_projects: %s\n" % e)

The response of OpenApiApi->get_projects:

[ApiProjectResponse(accessible=True, creation_date='2026-03-10T16:19:55.145394Z', current_user_id=403, deleted=False, id=79020, name='TFG_ADRIAN_ENCABO', owner_user_id=3)]


### Open a project and get the list of scenarios available

In [9]:
# My personal Project defined
MY_PROJECT_NAME="TFG_ADRIAN_ENCABO"
the_project_id=0
try:
    full_match = True # bool | Whether to match project name by complete match. (default to True)
    # Opens project with name projectName.
    the_project = api_instance.find_and_open_project_by_name(full_match, MY_PROJECT_NAME)
    print("The response of OpenApiApi->find_and_open_project_by_name:\n")
    pprint(the_project)
except Exception as e:
    print("Exception when calling OpenApiApi->find_and_open_project_by_name: %s\n" % e)
try:
    # Returns a list of scenarios for the project with identifier project_id.
    the_scenarios = api_instance.get_scenarios(the_project.id)
    print("The response of OpenApiApi->get_scenarios:\n")
    pprint(the_scenarios)
except Exception as e:
    print("Exception when calling OpenApiApi->get_scenarios: %s\n" % e)

The response of OpenApiApi->find_and_open_project_by_name:

ApiProjectResponse(accessible=True, creation_date='2026-03-10T16:19:55.145394Z', current_user_id=403, deleted=False, id=79020, name='TFG_ADRIAN_ENCABO', owner_user_id=3)
The response of OpenApiApi->get_scenarios:

[ApiScenarioData(id=82840, name='Budget Comparison (Estimated Demand)', type='SIM'),
 ApiScenarioData(id=87083, name='P3.3. GFA 1. Results 2_Different Locations', type='SIM'),
 ApiScenarioData(id=84191, name='Budget Comparison (Estimated Demand) 2', type='SIM'),
 ApiScenarioData(id=85538, name='Budget Comparison (Estimated Demand) 2', type='SIM'),
 ApiScenarioData(id=86671, name='P4.2.2 NO Distribution Network inside 4 Walls Models Capacity Restriction', type='NO'),
 ApiScenarioData(id=88185, name='P4.2.2 NO Distribution Network inside 4 Walls Models Capacity Restriction 2', type='NO'),
 ApiScenarioData(id=89212, name='P4.2.2 NO Distribution Network inside 4 Walls Models Capacity Restriction sim', type='SIM'),
 ApiSc

### Select the scenario by name

In [10]:
#Add the name of the scenario 
MY_SCENARIO_NAME="Global Network Examination"

# 1. Select the scenario
the_scenario = next(s for s in the_scenarios if s.name == MY_SCENARIO_NAME)
pprint(the_scenario)

ApiScenarioData(id=179567, name='Global Network Examination', type='SIM')


### Get the list of experiment runs of a scenario

In [11]:
try:
    # Returns the results of experiment runs for scenario.
    the_runs = api_instance.get_experiment_runs_for_scenario(the_scenario.id)
    print("The response of OpenApiApi->get_experiment_runs_for_scenario:\n")
    pprint(the_runs)
except Exception as e:
    print("Exception when calling OpenApiApi->get_experiment_runs_for_scenario: %s\n" % e)

The response of OpenApiApi->get_experiment_runs_for_scenario:

[ApiBasicRun(has_options=False, id=201331, name='Statistics', rc_id=201332, removing=False, scenario_id=179567, type='SIMULATION'),
 ApiBasicRun(has_options=False, id=201578, name='Statistics 2', rc_id=201579, removing=False, scenario_id=179567, type='SIMULATION'),
 ApiBasicRun(has_options=False, id=201796, name='Statistics 3', rc_id=201797, removing=False, scenario_id=179567, type='SIMULATION'),
 ApiBasicRun(has_options=False, id=217384, name='Statistics 4', rc_id=217385, removing=False, scenario_id=179567, type='SIMULATION'),
 ApiBasicRun(has_options=False, id=217581, name='Statistics 5', rc_id=217582, removing=False, scenario_id=179567, type='SIMULATION'),
 ApiBasicRun(has_options=False, id=220765, name='Statistics 6', rc_id=220766, removing=False, scenario_id=179567, type='SIMULATION'),
 ApiBasicRun(has_options=False, id=220963, name='Statistics 7', rc_id=220964, removing=False, scenario_id=179567, type='SIMULATION'),
 

## PART III. EXECUTE A SIMULATION

### Get the list of RunConfigurations for the experiments of a given Scenario
Each type of experiment has a RunConfiguration. For example:
```
ApiBasicRunConfiguration(animation_model_id=None, feasible_check_status='NOT_CONDUCTED', id=2439, is_feasible_check_data_available=False, model_time=None, name='i18n-Experiment_Simulation_text', progress=None, project_id=63, routes_progress=None, run_status=None, scenario_id=1752, speed=None, status='IDLE', step='IDLE', type='SIMULATION', validation_status='VALID'
```
Types of available experiment runs are:
* SIMULATION
* VARIATION
* COMPARISON
* SAFETY_STOCK
* RISK_ANALYSIS

In [12]:
MY_RUN_TYPE="SIMULATION"
try:
    # Returns a list of experiments available for a given scenario.
    the_run_configurations= api_instance.get_experiments(the_project.id, the_scenario.id)
    print("The response of OpenApiApi->get_experiments:\n")
    # Exception fixed for undefined variable
    pprint(the_run_configurations)
except Exception as e:
    print("Exception when calling OpenApiApi->get_experiments: %s\n" % e)

# 2. Select the adequate RunConfiguration for the scenario
the_RC = next((r for r in the_run_configurations if r.type == MY_RUN_TYPE), None)
pprint(the_RC)

The response of OpenApiApi->get_experiments:

[ApiBasicRunConfiguration(animation_model_id=None, feasible_check_status='NOT_CONDUCTED', id=180254, is_feasible_check_data_available=False, model_time=None, name='i18n-Experiment_Simulation_text', progress=None, project_id=79020, routes_progress=None, run_status=None, scenario_id=179567, speed=None, status='IDLE', step='IDLE', type='SIMULATION', validation_status='VALID'),
 ApiBasicRunConfiguration(animation_model_id=None, feasible_check_status='NOT_CONDUCTED', id=180429, is_feasible_check_data_available=False, model_time=None, name='i18n-Experiment_VariationExperiment_text', progress=None, project_id=79020, routes_progress=None, run_status=None, scenario_id=179567, speed=None, status='IDLE', step='IDLE', type='VARIATION', validation_status='NOT_CONDUCTED'),
 ApiBasicRunConfiguration(animation_model_id=None, feasible_check_status='NOT_CONDUCTED', id=179575, is_feasible_check_data_available=False, model_time=None, name='i18n-Experiment_Comp

### Run a new experiment over a scenario (synch)
Runs synchronously.


In [13]:
try:
    # Starts the experiment synchronously.
    the_sync_result = api_instance.run_experiment_synchronously(the_RC.id)
    print("The response of OpenApiApi->run_experiment_synchronously:\n")
    pprint(the_sync_result)
except Exception as e:
    print("Exception when calling OpenApiApi->run_experiment_synchronously: %s\n" % e)

The response of OpenApiApi->run_experiment_synchronously:

ApiExperimentResult(experiment_result_id=272060, validation_errors=None, validation_status='OK', validation_warnings=None)


## Part IIIb. Asynch execution

### Run a new experiment over a scenario (asynch)
Runs asynchronously.
Periodic status check.

In [14]:
skip_scenario_warnings = True # bool | Whether to skip scenario warnings. (optional)
try:
    # Starts the experiment asynchronously.
    the_execution = api_instance.run_experiment(the_RC.id, skip_scenario_warnings=skip_scenario_warnings)
    print("The response of OpenApiApi->run_experiment:\n")
    pprint(the_execution)
except Exception as e:
    print("Exception when calling OpenApiApi->run_experiment: %s\n" % e)

import time
while True:
    try:
        # Returns the experiment status
        the_status = api_instance.get_experiment_status(the_RC.id)
        # pprint(api_response)

        # Check if execution finished
        if the_status.finished:
            print("Execution finished.")
            break

        # Optional: stop if failed
        # fixed exception of status check
        if the_status.failed:
            print("Execution failed.")
            break

    except Exception as e:
        print("Exception when calling OpenApiApi->get_experiment_status:", e)
    print("Not finished yet. Wait 1sec and check again")
    # wait before next check
    time.sleep(1)

The response of OpenApiApi->run_experiment:

ApiRunExperiment(animation_model_id=180254, experiment_type_id='SIMULATION', is3d_animation=False)
Execution finished.


### Get the experiments results (pages)
You can ask for limited number of results (avoid overflow)

In [15]:
skip = 0 # int | Number of records to skip. (default to 0)
take = 1000 # int | Number of records to be retrieved. (default to 1000)

try:
    # Returns all the results of the specific experiment run.
    the_run_results = api_instance.get_experiment_run_result(the_RC.id, skip, take)
    print("The response of OpenApiApi->get_experiment_run_result:\n")
    pprint(the_run_results)
except Exception as e:
    print("Exception when calling OpenApiApi->get_experiment_run_result: %s\n" % e)


The response of OpenApiApi->get_experiment_run_result:

ApiExperimentResultData(pages=[ApiExperimentResultPageWrapper(charts=[ApiExperimentResultGraphChartWrapper(chart=ApiChartMetadataShort(id=180257, is_group_mode_enabled=False, layout=ApiChartLayoutData(height=1, start_col=0, start_row=0, width=2), name='Available Inventory', rc_id=180254, type='LINE'), class_name='ApiExperimentResultGraphChartWrapper', data=ApiGraphChartData(entity_list=[], id=180257, show_x=True, total_entity_count=0, total_technical_row_count=0, type='LINE')), ApiExperimentResultGraphChartWrapper(chart=ApiChartMetadataShort(id=180258, is_group_mode_enabled=False, layout=ApiChartLayoutData(height=1, start_col=0, start_row=1, width=2), name='Available Inventory Including Backlog', rc_id=180254, type='LINE'), class_name='ApiExperimentResultGraphChartWrapper', data=ApiGraphChartData(entity_list=[], id=180258, show_x=True, total_entity_count=0, total_technical_row_count=0, type='LINE')), ApiExperimentResultGraphChartW

In [16]:
for p in the_run_results.pages:
    print(f"\n{p.page.name} (id:{p.page.id})")
    for c in p.charts:
        print(f"  - {c.chart.name} (type: {c.chart.type}) (id: {c.chart.id})")


Available Inventory (id:180256)
  - Available Inventory (type: LINE) (id: 180257)
  - Available Inventory Including Backlog (type: LINE) (id: 180258)
  - Average Daily Available Inventory (type: LINE) (id: 180259)
  - Average Daily On-hand Inventory (type: LINE) (id: 180260)
  - On-hand Inventory (type: LINE) (id: 180261)

Fulfillment (id:180262)
  - Demand Placed, Fulfillment Received (Accumulative) (type: TABLE) (id: 180263)
  - Demand Placed, Fulfillment Received (Accumulative, Per Object) (type: TABLE) (id: 180264)
  - Demand Placed, Fulfillment Received (Daily) (type: LINE) (id: 180265)
  - Demand Placed, Fulfillment Received (Daily, Per Object) (type: LINE) (id: 180266)
  - Demand Received, Fulfillment Shipped (Accumulative) (type: TABLE) (id: 180267)
  - Demand Received, Fulfillment Shipped (Daily) (type: LINE) (id: 180268)

Lead Time (id:180269)
  - Lead Time (type: LINE) (id: 180270)
  - Max Lead Time (type: TABLE) (id: 180271)
  - Mean Lead Time (Best-Mean-Worst) (type: BMW-

### Obtain the pages in the experiment's result dashboard

In [17]:
try:
    # Returns statistics pages for the result of experiment run.
    # Exception fixed for 'id' attribute
    # We use the experiment_result_id from the previously executed synchronous run
    result_id = the_sync_result.experiment_result_id
    dashboard_pages = api_instance.get_experiment_dashboard_pages(result_id)
    print("The response of OpenApiApi->get_experiment_dashboard_pages:\n")
    
    # CHANGED Displaying the available pages and charts with their dynamic IDs
    for page in dashboard_pages:
        print(f"Dashboard Page: {page.name} (ID: {page.id})")
        for chart in page.charts:
            print(f"  - Chart: {chart.name} (ID: {chart.id})")
            
except Exception as e:
    print("Exception when calling OpenApiApi->get_experiment_dashboard_pages: %s\n" % e)

The response of OpenApiApi->get_experiment_dashboard_pages:

Dashboard Page: Available Inventory (ID: 272062)
  - Chart: Available Inventory (ID: 272063)
  - Chart: Available Inventory Including Backlog (ID: 272064)
  - Chart: Average Daily Available Inventory (ID: 272065)
  - Chart: Average Daily On-hand Inventory (ID: 272066)
  - Chart: On-hand Inventory (ID: 272067)
Dashboard Page: Fulfillment (ID: 272068)
  - Chart: Demand Placed, Fulfillment Received (Accumulative) (ID: 272069)
  - Chart: Demand Placed, Fulfillment Received (Accumulative, Per Object) (ID: 272070)
  - Chart: Demand Placed, Fulfillment Received (Daily) (ID: 272071)
  - Chart: Demand Placed, Fulfillment Received (Daily, Per Object) (ID: 272072)
  - Chart: Demand Received, Fulfillment Shipped (Accumulative) (ID: 272073)
  - Chart: Demand Received, Fulfillment Shipped (Daily) (ID: 272074)
Dashboard Page: Lead Time (ID: 272075)
  - Chart: Lead Time (ID: 272076)
  - Chart: Max Lead Time (ID: 272077)
  - Chart: Mean Lead 

### Export the results as an excel matrix

In [18]:
# Exception fixed for wrong IDs
# EXPORT ALL DASHBOARD PAGES TO EXCEL
try:
    # Dynamically select the first dashboard page to export to avoid hardcoded ID errors
    if dashboard_pages and len(dashboard_pages) > 0:
        target_result_id = the_sync_result.experiment_result_id
        print(f"Exporting ALL dashboard pages from result ID {target_result_id}...\n")
        
        # Iterate over every page available in the dashboard
        for page in dashboard_pages:
            
            # Clean the page name to make it a safe Windows filename (replace spaces with underscores)
            safe_page_name = re.sub(r'[^a-zA-Z0-9_\-]', '_', page.name)

            # Save the exported Excel file locally
            # Create a unique filename for each page
            output_filename = f"Simulation_Results_{safe_page_name}.xlsx"
            output_path = os.path.join(os.getcwd(), output_filename)
            
            print(f"Downloading '{page.name}' (ID: {page.id}) -> {output_filename}")

            # Returns an Excel representation of the dashboard page
            # Export this specific page
            excel_export = api_instance.export_dashboard_page(target_result_id, page.id)

            # Check if the API returns a temporary file path
            # Save the file securely
            if isinstance(excel_export, str) and os.path.exists(excel_export):
                shutil.move(excel_export, output_path)
            elif isinstance(excel_export, bytes):
                with open(output_path, "wb") as file:
                    file.write(excel_export)
            else:
                print(f"Unexpected data format received for page {page.name}")
                
        print("\n[SUCCESS] All dashboard pages exported successfully.")
    else:
        print("No dashboard pages available to export.")
        
except Exception as e:
    print("Exception when calling OpenApiApi->export_dashboard_page: %s\n" % e)

Exporting ALL dashboard pages from result ID 272060...


[SUCCESS] All dashboard pages exported successfully.


### Close current project

In [19]:
#Error fixed of indentation
try:
    # Closes the currently open project.
    api_response = api_instance.close_project()
    print("The response of OpenApiApi->close_project:\n")
    pprint(api_response)
    print("\nProject closed successfully.")
except ApiException as e:
    print("Exception when calling OpenApiApi->close_project: %s\n" % e)

The response of OpenApiApi->close_project:

ApiProjectResponse(accessible=True, creation_date='2026-03-10T16:19:55.145394Z', current_user_id=None, deleted=False, id=79020, name='TFG_ADRIAN_ENCABO', owner_user_id=3)

Project closed successfully.


## ==========================================================
## PHASE 2 Flat AI Decision & Scenario Cloning

## Install the dependencies (All required for PHASE 2)

In [20]:
# Install xlwings to interact natively with Microsoft Excel
!{sys.executable} -m pip install xlwings

In [ ]:
import os
import shutil
import xlwings as xw  #For excel modifications
import time
from pprint import pprint

import re  # Used for cleaning text strings


### 1. Define the Flat AI logic (3 predefined decisions)

In [26]:
decision_options = [
    "DECISION 0: Increase Demand by 20% (Aggressive Growth)",
    "DECISION 1: Decrease Transport Costs by 15% (Cost Optimization)",
    "DECISION 2: Increase Safety Stock by 10% (Conservative Risk Management)"
]

print("--- FLAT AI: EVALUATING PREDEFINED DECISIONS ---")
for i, decision in enumerate(decision_options):
    print(f"[{i}] {decision}")

# For this 'flat AI' phase, we will simulate the AI choosing a decision automatically.
# Let us assume the AI evaluates the current context and selects Decision 0.
selected_decision_index = 2

selected_decision_text = decision_options[selected_decision_index]

print(f"\nFLAT AI SELECTED: {selected_decision_text}")

--- FLAT AI: EVALUATING PREDEFINED DECISIONS ---
[0] DECISION 0: Increase Demand by 20% (Aggressive Growth)
[1] DECISION 1: Decrease Transport Costs by 15% (Cost Optimization)
[2] DECISION 2: Increase Safety Stock by 10% (Conservative Risk Management)

FLAT AI SELECTED: DECISION 2: Increase Safety Stock by 10% (Conservative Risk Management)


### 2. Flat AI Autonomous Data Modification

In [27]:
# Define the paths for the original and the new modified scenario
ORIGINAL_EXCEL_FILENAME = "Global Network Examination.xlsx"
ORIGINAL_EXCEL_PATH = os.path.abspath(ORIGINAL_EXCEL_FILENAME)
MODIFIED_EXCEL_PATH = os.path.abspath(f"Modified_{ORIGINAL_EXCEL_FILENAME}")

print(f"Reading original scenario from: {ORIGINAL_EXCEL_PATH}")

# 1. Create a fresh copy of the original Excel to avoid corrupting the base one
try:
    shutil.copy2(ORIGINAL_EXCEL_PATH, MODIFIED_EXCEL_PATH)
    print("Created a clean copy of the scenario.")
except Exception as e:
    print(f"Error copying file: {e}")

print("Opening Microsoft Excel in the background to preserve strict formatting...")

# 2. Open Microsoft Excel silently (visible=False) so it doesn't interrupt you
app = xw.App(visible=False) 

try:
    wb = app.books.open(MODIFIED_EXCEL_PATH)
    
    if selected_decision_index == 0:
        # DECISION 0: Increase Demand by 20%
        print(f"Applying {decision_options[selected_decision_index]}...")
        ws = wb.sheets['Demand']
        
        # Get the range of used columns and rows
        last_col = ws.used_range.last_cell.column
        last_row = ws.used_range.last_cell.row
        
        # Extract headers to find the right columns dynamically
        headers_raw = ws.range((1, 1), (1, last_col)).value
        headers = headers_raw if isinstance(headers_raw, list) else [headers_raw]
        
        if 'Col 7' in headers and 'Col 10' in headers:
            col7_idx = headers.index('Col 7') + 1
            col10_idx = headers.index('Col 10') + 1
            
            for r in range(2, last_row + 1):
                if ws.range((r, col7_idx)).value == 'Quantity':
                    current_val = ws.range((r, col10_idx)).value
                    if isinstance(current_val, (int, float)):
                        ws.range((r, col10_idx)).value = current_val * 1.20
            print("Demand increased by 20% successfully.")
            
    elif selected_decision_index == 1:
        # DECISION 1: Decrease Transport Costs by 15%
        print(f"Applying {decision_options[selected_decision_index]}...")
        ws = wb.sheets['Paths']
        
        last_col = ws.used_range.last_cell.column
        last_row = ws.used_range.last_cell.row
        
        headers_raw = ws.range((1, 1), (1, last_col)).value
        headers = headers_raw if isinstance(headers_raw, list) else [headers_raw]
        
        if 'Col 3' in headers and 'Col 4' in headers:
            col3_idx = headers.index('Col 3') + 1
            col4_idx = headers.index('Col 4') + 1
            
            for r in range(2, last_row + 1):
                val3 = ws.range((r, col3_idx)).value
                if val3 in ['Cost per unit', 'Cost']:
                    current_val = ws.range((r, col4_idx)).value
                    if isinstance(current_val, (int, float)):
                        ws.range((r, col4_idx)).value = current_val * 0.85
            print("Transport costs decreased by 15% successfully.")

    elif selected_decision_index == 2:
        # DECISION 2: Increase Safety Stock by 10%
        print(f"Applying {decision_options[selected_decision_index]}...")
        ws = wb.sheets['Inventory']
        
        last_col = ws.used_range.last_cell.column
        last_row = ws.used_range.last_cell.row
        
        headers_raw = ws.range((1, 1), (1, last_col)).value
        headers = headers_raw if isinstance(headers_raw, list) else [headers_raw]
        
        if 'Col 5' in headers and 'Col 6' in headers:
            col5_idx = headers.index('Col 5') + 1
            col6_idx = headers.index('Col 6') + 1
            
            for r in range(2, last_row + 1):
                if ws.range((r, col5_idx)).value == 'Safety stock':
                    current_val = ws.range((r, col6_idx)).value
                    if isinstance(current_val, (int, float)):
                        ws.range((r, col6_idx)).value = current_val * 1.10
            print("Safety stock increased by 10% successfully.")

    # 3. Save changes explicitly using native Microsoft Excel
    wb.save()
    print(f"\nModified scenario safely saved to: {MODIFIED_EXCEL_PATH}")
    print("Ready for ALX API Import.")

except Exception as e:
    print(f"An error occurred during native Excel modification: {e}")
finally:
    # 4. Always ensure the hidden Excel application is closed, even if it crashes
    try:
        wb.close()
    except: pass
    app.quit()

Reading original scenario from: C:\Users\Luis\Downloads\TFG\API\Global Network Examination.xlsx
Created a clean copy of the scenario.
Opening Microsoft Excel in the background to preserve strict formatting...
Applying DECISION 2: Increase Safety Stock by 10% (Conservative Risk Management)...
Safety stock increased by 10% successfully.

Modified scenario safely saved to: C:\Users\Luis\Downloads\TFG\API\Modified_Global Network Examination.xlsx
Ready for ALX API Import.


### 3. Modification by Excel

In [28]:
# Create a unique name for the new scenario based on the AI decision
new_scenario_name = f"Modified_AI_Decision_{selected_decision_index}"
print(f"Importing '{MODIFIED_EXCEL_PATH}' as new scenario: '{new_scenario_name}'...")

try:
    # We use import_excel to upload the file and create a new scenario.
    # Note: In most Python OpenAPI clients, passing the file path as a string is the correct way to upload a file.
    import_response = api_instance.import_excel(
        new_scenario_name=new_scenario_name,
        project_id=the_project.id,
        file=MODIFIED_EXCEL_PATH,
        need_to_import_experiments=True # We want to keep the experiments configured
    )
    
    print("Import process initiated successfully.")
    pprint(import_response)
    
    # Since import_excel is asynchronous, we need to track the job_id until it finishes
    if import_response.job_id:
        print(f"\nTracking import job ID: {import_response.job_id}...")
        
        while True:
            # Get the current status of the import job
            import_status = api_instance.get_import_status(import_response.job_id)
            print(f"Current import status: {import_status.status}")
            
            # Check if the process has reached a final state
            if import_status.status in ['DONE', 'FAILED', 'CANCELED']:
                print(f"Import process finished with status: {import_status.status}")
                
                if import_status.status == 'DONE':
                    # We save the ID of the newly created scenario for the next steps
                    new_scenario_id = import_status.scenario_id
                    print(f"\nSUCCESS! New scenario created with ID: {new_scenario_id}")
                else:
                    # In case it fails, it usually provides an error message
                    print(f"Error details (if any): {getattr(import_status, 'error', 'No error details provided')}")
                break
            
            # Wait for 2 seconds before checking again
            time.sleep(2)
    else:
        print("Warning: No job_id received from the import response.")
            
except Exception as e:
    print(f"Exception when calling OpenApiApi->import_excel: {e}")

Importing 'C:\Users\Luis\Downloads\TFG\API\Modified_Global Network Examination.xlsx' as new scenario: 'Modified_AI_Decision_2'...
Import process initiated successfully.
ApiImportResponse(error=None, job_id=120)

Tracking import job ID: 120...
Current import status: DONE
Import process finished with status: DONE

SUCCESS! New scenario created with ID: 275557


### 4. Run simulation on AI scenario

In [31]:
# Run simulation on the new AI scenario
# The AI automatically configures a SIMULATION experiment for its newly created scenario, 
# and runs it synchronously to generate the results.

print(f"--- PHASE 2 - BLOCK 4: RUNNING SIMULATION FOR NEW AI SCENARIO (ID: {new_scenario_id}) ---")

try:
    print("1. Fetching SIMULATION configuration for the imported scenario...")
    # Retrieve experiments explicitly for the AI scenario
    ai_run_configurations = api_instance.get_experiments(the_project.id, new_scenario_id)
    
    # Select the 'SIMULATION' type experiment
    ai_simulation_RC = next((r for r in ai_run_configurations if r.type == 'SIMULATION'), None)
    
    if ai_simulation_RC:
        print(f"Found SIMULATION experiment (ID: {ai_simulation_RC.id}). Launching synchronously...")
        
        # 2. Run the simulation
        sim_result = api_instance.run_experiment_synchronously(ai_simulation_RC.id)
        # Safely extract the result ID
        the_ai_sim_result_id = getattr(sim_result, 'experiment_result_id', None)
        
        if the_ai_sim_result_id:
            print(f"\n[SYSTEM] Simulation finished! Captured Result ID: {the_ai_sim_result_id}")

            # 3. Fetch the full list of dashboard pages (KPIs) and store them for the next block
            print("\n3. Fetching dashboard pages layout...")
            ai_dashboard_pages = api_instance.get_experiment_dashboard_pages(the_ai_sim_result_id)
            print(f"Found {len(ai_dashboard_pages)} dashboard pages ready for export.")
            
        else:
            print("Failed to capture 'experiment_result_id' from the synchronous run.")
            pprint(sim_result)
    else:
        print("No SIMULATION experiment found for the AI scenario.")
        
except Exception as e:
    print("Exception during Simulation Execution: %s\n" % e)

--- PHASE 2 - BLOCK 4: RUNNING SIMULATION FOR NEW AI SCENARIO (ID: 275557) ---
1. Fetching SIMULATION configuration for the imported scenario...
Found SIMULATION experiment (ID: 276244). Launching synchronously...

[SYSTEM] Simulation finished! Captured Result ID: 276773

3. Fetching dashboard pages layout...
Found 7 dashboard pages ready for export.


### 5. Export the new dashboard pages

In [32]:
# Export results as an Excel matrix
# The AI loops through every available dashboard page (Finance, Inventory, etc.)
# and saves each one as a separate Excel file to preserve visual charts.

print(f"--- PHASE 2 - BLOCK 5: EXPORTING AI SIMULATION RESULTS ---")

try:
    if 'ai_dashboard_pages' in locals() and len(ai_dashboard_pages) > 0:
        print(f"Exporting ALL dashboard pages from AI result ID {the_ai_sim_result_id}...\n")

        # Iterate over every page available in the dashboard
        for page in ai_dashboard_pages:
            # Clean the page name to make it a safe Windows filename (replace spaces with underscores)
            safe_page_name = re.sub(r'[^a-zA-Z0-9_\-]', '_', page.name)
            
            # Use the selected_decision_index to name the file accordingly
            # Create a unique filename for each page, identifying it's from the AI decision
            output_filename = f"AI_Decision{selected_decision_index}_Results_{safe_page_name}.xlsx"
            output_path = os.path.join(os.getcwd(), output_filename)
            
            print(f"Downloading '{page.name}' (ID: {page.id}) -> {output_filename}")

            # Export this specific page
            excel_export = api_instance.export_dashboard_page(the_ai_sim_result_id, page.id)
            
            # Save the file securely (handle both string paths and byte streams)
            if isinstance(excel_export, str) and os.path.exists(excel_export):
                shutil.move(excel_export, output_path)
            elif isinstance(excel_export, bytes):
                with open(output_path, "wb") as file:
                    file.write(excel_export)
            else:
                print(f"Unexpected data format received for page {page.name}")
                
        print("\n[TOTAL SUCCESS] All AI dashboard pages exported successfully.")
    else:
        print("No dashboard pages available to export.")
        
except Exception as e:
    print("Exception when exporting AI dashboard pages: %s\n" % e)

--- PHASE 2 - BLOCK 5: EXPORTING AI SIMULATION RESULTS ---
Exporting ALL dashboard pages from AI result ID 276773...


[TOTAL SUCCESS] All AI dashboard pages exported successfully.


### ==========================================================
## DEVELOPMENT TESTS - IGNORE THEM

In [ ]:
#Avoid self-signed warnings 
import os
import urllib3

For self-signed certificates it is necesary to donwload them and add to the certification path

In [ ]:
import ssl
import socket

def download_self_signed_cert_no_validation(hostname, port, cert_file):
    context = ssl._create_unverified_context()
    conn = context.wrap_socket(socket.socket(socket.AF_INET), server_hostname=hostname)
    conn.connect((hostname, port))
    der_cert = conn.getpeercert(binary_form=True)
    pem_cert = ssl.DER_cert_to_PEM_cert(der_cert)
    
    with open(cert_file, 'w') as cert_file:
        cert_file.write(pem_cert)
    
    print(f"Certificate saved to {cert_file.name}")

# Example usage
download_self_signed_cert_no_validation(SERVER_IP, 443, MY_CERT_FILENAME)

In [ ]:
# !/usr/bin/openssl s_client -showcerts -connect {SERVER_IP}:443 </dev/null 2>/dev/null | sed -n -e '/BEGIN\ CERTIFICATE/,/ENDCERTIFICATE/ p' > {MY_CERT_FILE}

In [ ]:


MY_CERT_FILE=os.path.abspath(MY_CERT_FILENAME)
if os.path.isfile(MY_CERT_FILE):
    print(f"Cert file OK: {MY_CERT_FILE} ")
else:
    print(f"Cert file ERROR: {MY_CERT_FILE} ")
MY_CERT_FILE

In [ ]:
os.environ['REQUESTS_CA_BUNDLE'] = MY_CERT_FILE

# Create a PoolManager with a custom CA bundle
http = urllib3.PoolManager(
    cert_reqs='CERT_NONE',      # Enforce cert verification
    ca_certs=MY_CERT_FILE           # Path to your self-signed cert
)

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
http.request("GET", TEST_URL)

In [ ]:
from urllib3 import HTTPSConnectionPool

pool = HTTPSConnectionPool(
    SERVER_IP,
    port=443,
    cert_reqs='CERT_REQUIRED',
    ca_certs=MY_CERT_FILE
)

In [ ]:
import requests 

# Any request you make now will use your certificate
response = requests.get(TEST_URL, verify=False)
print(response.text)